# Cài đặt thuật toán Canny Edge Detection & Tối ưu hóa Vectorization / Numba

**Mục tiêu:**
1. Tự cài đặt chi tiết 4 bước của thuật toán Canny: `Gaussian Blur` $\rightarrow$ `Sobel Gradient` $\rightarrow$ `Non-Maximum Suppression (NMS)` $\rightarrow$ `Hysteresis Thresholding`.
2. **Tối ưu hóa hiệu năng:** So sánh vòng lặp chuẩn (Nested Loops), Vectorization (NumPy slicing) và JIT Compilation (Numba).
3. So sánh đối chiếu với hàm `cv2.Canny()` và Scikit-image `skimage.feature.canny()`.
4. **Thử nghiệm mở rộng:** Đánh giá trên nhiều loại ảnh (Chuẩn, Tương phản thấp, Nhiễu, Chi tiết phức tạp) và các bộ ngưỡng Hysteresis khác nhau.

In [ ]:
import os
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import binary_dilation
from skimage import feature

try:
    from numba import jit
    HAS_NUMBA = True
except ImportError:
    HAS_NUMBA = False

print(f"OpenCV version: {cv2.__version__}")
print(f"Numba available: {HAS_NUMBA}")

## Bước 0: Đọc ảnh & Chuyển sang ảnh xám (Grayscale)

In [ ]:
# Tim va doc anh mau
candidate_paths = [
    'circle.jpg',
    os.path.join('..', 'circle.jpg'),
    os.path.join('..', 'Lap1', 'circle.jpg'),
    os.path.join('..', 'input', 'circle.jpg')
]

image_path = None
for path in candidate_paths:
    if os.path.exists(path):
        image_path = path
        break

if image_path:
    image = cv2.imread(image_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
else:
    # Tao anh mau hinh hoc tong hop neu khong tim thay file anh ngoai
    gray = np.full((300, 300), 230, dtype=np.uint8)
    cv2.circle(gray, (150, 150), 70, (40,), -1)
    cv2.rectangle(gray, (40, 40), (100, 100), (90,), -1)
    cv2.line(gray, (30, 260), (270, 260), (20,), 4)

print(f"Kich thuoc anh xam: {gray.shape}")

## Bước 1: Gaussian Blur (Lọc làm mượt, giảm nhiễu)

In [ ]:
blurred = cv2.GaussianBlur(gray, (5, 5), sigmaX=1.4)

## Bước 2: Tính Gradient bằng toán tử Sobel (Độ lớn Magnitude & Hướng Angle)

In [ ]:
sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
magnitude = np.sqrt(sobel_x**2 + sobel_y**2).astype(np.float32)
angle = (np.arctan2(sobel_y, sobel_x) * 180 / np.pi).astype(np.float32)
angle[angle < 0] += 180  # Dua ve mien [0, 180)

## Bước 3: Non-Maximum Suppression (NMS - Làm mảnh biên)
So sánh 3 cách cài đặt:
1. **Vòng lặp tuần tự (Nested Loop)**
2. **Vectorization bằng NumPy Slicing & Boolean Masking**
3. **JIT Compilation với Numba**

In [ ]:
def nms_loop(magnitude, angle):
    H, W = magnitude.shape
    result = np.zeros((H, W), dtype=np.float32)
    for i in range(1, H - 1):
        for j in range(1, W - 1):
            a = angle[i, j]
            if (0 <= a < 22.5) or (157.5 <= a <= 180):
                n1, n2 = magnitude[i, j+1], magnitude[i, j-1]
            elif 22.5 <= a < 67.5:
                n1, n2 = magnitude[i+1, j-1], magnitude[i-1, j+1]
            elif 67.5 <= a < 112.5:
                n1, n2 = magnitude[i+1, j], magnitude[i-1, j]
            else:
                n1, n2 = magnitude[i-1, j-1], magnitude[i+1, j+1]

            if magnitude[i, j] >= n1 and magnitude[i, j] >= n2:
                result[i, j] = magnitude[i, j]
    return result

def nms_vectorized(magnitude, angle):
    H, W = magnitude.shape
    result = np.zeros((H, W), dtype=np.float32)
    mag_c = magnitude[1:-1, 1:-1]
    ang_c = angle[1:-1, 1:-1]
    q = np.zeros_like(mag_c)
    r = np.zeros_like(mag_c)

    # 0 deg (E-W)
    mask_0 = ((0 <= ang_c) & (ang_c < 22.5)) | ((157.5 <= ang_c) & (ang_c <= 180))
    q[mask_0] = magnitude[1:-1, 2:][mask_0]
    r[mask_0] = magnitude[1:-1, :-2][mask_0]

    # 45 deg (NE-SW)
    mask_45 = (22.5 <= ang_c) & (ang_c < 67.5)
    q[mask_45] = magnitude[2:, :-2][mask_45]
    r[mask_45] = magnitude[:-2, 2:][mask_45]

    # 90 deg (N-S)
    mask_90 = (67.5 <= ang_c) & (ang_c < 112.5)
    q[mask_90] = magnitude[2:, 1:-1][mask_90]
    r[mask_90] = magnitude[:-2, 1:-1][mask_90]

    # 135 deg (NW-SE)
    mask_135 = (112.5 <= ang_c) & (ang_c < 157.5)
    q[mask_135] = magnitude[:-2, :-2][mask_135]
    r[mask_135] = magnitude[2:, 2:][mask_135]

    keep = (mag_c >= q) & (mag_c >= r)
    result[1:-1, 1:-1][keep] = mag_c[keep]
    return result

if HAS_NUMBA:
    @jit(nopython=True, fastmath=True)
    def nms_numba(magnitude, angle):
        H, W = magnitude.shape
        result = np.zeros((H, W), dtype=np.float32)
        for i in range(1, H - 1):
            for j in range(1, W - 1):
                a = angle[i, j]
                if (0 <= a < 22.5) or (157.5 <= a <= 180):
                    n1, n2 = magnitude[i, j+1], magnitude[i, j-1]
                elif 22.5 <= a < 67.5:
                    n1, n2 = magnitude[i+1, j-1], magnitude[i-1, j+1]
                elif 67.5 <= a < 112.5:
                    n1, n2 = magnitude[i+1, j], magnitude[i-1, j]
                else:
                    n1, n2 = magnitude[i-1, j-1], magnitude[i+1, j+1]

                if magnitude[i, j] >= n1 and magnitude[i, j] >= n2:
                    result[i, j] = magnitude[i, j]
        return result
else:
    nms_numba = nms_vectorized

nms = nms_vectorized(magnitude, angle)
print("NMS Vectorized da thuc hien thanh cong!")

## Bước 4: Hysteresis Thresholding & Edge Tracking (Nối biên hai ngưỡng)
So sánh:
1. **Vòng lặp 8 láng giềng**
2. **Vectorization Dilation (Morphological Connected Components)**
3. **DFS Stack Tracking với Numba**

In [ ]:
def hysteresis_loop(img, low_ratio=0.05, high_ratio=0.15):
    high_threshold = img.max() * high_ratio
    low_threshold = high_threshold * low_ratio
    H, W = img.shape
    result = np.zeros((H, W), dtype=np.uint8)
    strong, weak = 255, 75

    result[img >= high_threshold] = strong
    result[(img >= low_threshold) & (img < high_threshold)] = weak

    for i in range(1, H - 1):
        for j in range(1, W - 1):
            if result[i, j] == weak:
                if strong in (result[i-1,j-1], result[i-1,j], result[i-1,j+1],
                              result[i,j-1],               result[i,j+1],
                              result[i+1,j-1], result[i+1,j], result[i+1,j+1]):
                    result[i, j] = strong
                else:
                    result[i, j] = 0
    return result

def hysteresis_vectorized(img, low_ratio=0.05, high_ratio=0.15):
    high_threshold = img.max() * high_ratio
    low_threshold = high_threshold * low_ratio

    strong_edges = img >= high_threshold
    weak_edges = (img >= low_threshold) & (img < high_threshold)
    struct_8 = np.ones((3, 3), dtype=bool)

    connected = strong_edges.copy()
    while True:
        dilated = binary_dilation(connected, structure=struct_8)
        new_connected = dilated & weak_edges
        if not np.any(new_connected & (~connected)):
            break
        connected |= new_connected

    result = np.zeros(img.shape, dtype=np.uint8)
    result[connected] = 255
    return result

if HAS_NUMBA:
    @jit(nopython=True, fastmath=True)
    def hysteresis_numba(img, low_ratio=0.05, high_ratio=0.15):
        high_threshold = img.max() * high_ratio
        low_threshold = high_threshold * low_ratio
        H, W = img.shape
        result = np.zeros((H, W), dtype=np.uint8)
        stack_r = np.zeros(H * W, dtype=np.int32)
        stack_c = np.zeros(H * W, dtype=np.int32)
        top = 0

        for r in range(H):
            for c in range(W):
                val = img[r, c]
                if val >= high_threshold:
                    result[r, c] = 255
                    stack_r[top] = r
                    stack_c[top] = c
                    top += 1
                elif val >= low_threshold:
                    result[r, c] = 75

        while top > 0:
            top -= 1
            cr = stack_r[top]
            cc = stack_c[top]
            for dr in (-1, 0, 1):
                for dc in (-1, 0, 1):
                    if dr == 0 and dc == 0:
                        continue
                    nr, nc = cr + dr, cc + dc
                    if 0 <= nr < H and 0 <= nc < W:
                        if result[nr, nc] == 75:
                            result[nr, nc] = 255
                            stack_r[top] = nr
                            stack_c[top] = nc
                            top += 1

        for r in range(H):
            for c in range(W):
                if result[r, c] == 75:
                    result[r, c] = 0
        return result
else:
    hysteresis_numba = hysteresis_vectorized

edges_manual = hysteresis_vectorized(nms, low_ratio=0.05, high_ratio=0.15)
print("Hysteresis da thuc hien thanh cong!")

## Benchmark hiệu năng: Loop vs NumPy Vectorized vs Numba JIT vs OpenCV

In [ ]:
# Warmup Numba
if HAS_NUMBA:
    _ = nms_numba(magnitude, angle)
    _ = hysteresis_numba(magnitude, 0.05, 0.15)

def benchmark_method(name, nms_fn, hys_fn, runs=5):
    t_nms, t_hys = 0.0, 0.0
    for _ in range(runs):
        t0 = time.perf_counter()
        nms_out = nms_fn(magnitude, angle)
        t1 = time.perf_counter()
        _ = hys_fn(nms_out, 0.05, 0.15)
        t2 = time.perf_counter()
        t_nms += (t1 - t0)
        t_hys += (t2 - t1)
    avg_nms = (t_nms / runs) * 1000
    avg_hys = (t_hys / runs) * 1000
    print(f"[{name:25s}] NMS: {avg_nms:6.2f} ms | Hysteresis: {avg_hys:6.2f} ms | Tổng: {avg_nms + avg_hys:6.2f} ms")

print("=" * 65)
print(f"BENCHMARK HIỆU NĂNG ({gray.shape[1]}x{gray.shape[0]} px)")
print("=" * 65)
benchmark_method("1. Nested Loop", nms_loop, hysteresis_loop)
benchmark_method("2. NumPy Vectorized", nms_vectorized, hysteresis_vectorized)
if HAS_NUMBA:
    benchmark_method("3. Numba JIT (Compiled)", nms_numba, hysteresis_numba)

t_cv = 0.0
for _ in range(5):
    t0 = time.perf_counter()
    _ = cv2.Canny(blurred, 50, 150)
    t_cv += (time.perf_counter() - t0)
print(f"[{'4. OpenCV cv2.Canny()':25s}] Tổng: {(t_cv / 5)*1000:6.2f} ms (Tối ưu C++)")
print("=" * 65)

## Đối chiếu kết quả: Tự cài đặt vs OpenCV vs Scikit-image

In [ ]:
edges_opencv = cv2.Canny(blurred, 50, 150)
edges_skimage = feature.canny(gray, sigma=1.4, low_threshold=20, high_threshold=60)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes[0,0].imshow(gray, cmap='gray'); axes[0,0].set_title('0. Ảnh xám gốc'); axes[0,0].axis('off')
axes[0,1].imshow(blurred, cmap='gray'); axes[0,1].set_title('1. Gaussian Blur'); axes[0,1].axis('off')
axes[0,2].imshow(np.uint8(255*magnitude/np.max(magnitude)), cmap='gray'); axes[0,2].set_title('2. Sobel Magnitude'); axes[0,2].axis('off')
axes[1,0].imshow(np.uint8(255*nms/(np.max(nms)+1e-8)), cmap='gray'); axes[1,0].set_title('3. NMS (Vectorized)'); axes[1,0].axis('off')
axes[1,1].imshow(edges_manual, cmap='gray'); axes[1,1].set_title('4. Tự cài đặt (Hysteresis)'); axes[1,1].axis('off')
axes[1,2].imshow(edges_opencv, cmap='gray'); axes[1,2].set_title('Đối chiếu: cv2.Canny()'); axes[1,2].axis('off')

plt.tight_layout()
os.makedirs('output', exist_ok=True)
plt.savefig('output/canny_steps.png', dpi=150, bbox_inches='tight')
plt.show()

## Thử nghiệm mở rộng trên nhiều loại ảnh & các bộ ngưỡng khác nhau

In [ ]:
# Tao tap anh thu nghiem: Chuan, Tuong phan thap, Nhieu, Chi tiet phuc tap
img_shapes = np.full((300, 300), 230, dtype=np.uint8)
cv2.circle(img_shapes, (150, 150), 70, (40,), -1)
cv2.rectangle(img_shapes, (40, 40), (100, 100), (90,), -1)
cv2.line(img_shapes, (30, 260), (270, 260), (20,), 4)

img_low_contrast = (img_shapes.astype(np.float32) * 0.25 + 100).clip(0, 255).astype(np.uint8)

img_noisy = img_shapes.copy().astype(np.float32)
img_noisy = np.clip(img_noisy + np.random.normal(0, 25, img_noisy.shape), 0, 255).astype(np.uint8)

x = np.linspace(-3, 3, 300); y = np.linspace(-3, 3, 300)
xx, yy = np.meshgrid(x, y)
img_texture = ((np.sin(xx**2 + yy**2) + 1) * 127.5).astype(np.uint8)

test_set = [
    ("Hình khối chuẩn", img_shapes),
    ("Độ tương phản thấp", img_low_contrast),
    ("Ảnh nhiễu Gaussian", img_noisy),
    ("Vân sóng giao thoa", img_texture)
]

fig, axes = plt.subplots(4, 4, figsize=(18, 16))
for idx, (title, img) in enumerate(test_set):
    b1 = cv2.GaussianBlur(img, (5, 5), 1.4)
    b2 = cv2.GaussianBlur(img, (7, 7), 2.5)
    
    sx1 = cv2.Sobel(b1, cv2.CV_64F, 1, 0); sy1 = cv2.Sobel(b1, cv2.CV_64F, 0, 1)
    mag1 = np.sqrt(sx1**2 + sy1**2); ang1 = np.arctan2(sy1, sx1)*180/np.pi; ang1[ang1<0]+=180
    res1 = hysteresis_vectorized(nms_vectorized(mag1, ang1), 0.05, 0.15)

    sx2 = cv2.Sobel(b2, cv2.CV_64F, 1, 0); sy2 = cv2.Sobel(b2, cv2.CV_64F, 0, 1)
    mag2 = np.sqrt(sx2**2 + sy2**2); ang2 = np.arctan2(sy2, sx2)*180/np.pi; ang2[ang2<0]+=180
    res2 = hysteresis_vectorized(nms_vectorized(mag2, ang2), 0.05, 0.15)

    res_cv = cv2.Canny(b1, 50, 150)

    axes[idx, 0].imshow(img, cmap='gray'); axes[idx, 0].set_title(f"Gốc: {title}"); axes[idx, 0].axis('off')
    axes[idx, 1].imshow(res1, cmap='gray'); axes[idx, 1].set_title("Canny Tự cài (sigma=1.4)"); axes[idx, 1].axis('off')
    axes[idx, 2].imshow(res2, cmap='gray'); axes[idx, 2].set_title("Canny Lọc mạnh (sigma=2.5)"); axes[idx, 2].axis('off')
    axes[idx, 3].imshow(res_cv, cmap='gray'); axes[idx, 3].set_title("OpenCV cv2.Canny()"); axes[idx, 3].axis('off')

plt.tight_layout()
plt.savefig('output/canny_image_types_test.png', dpi=150, bbox_inches='tight')
plt.show()